# EDA LFP — Pré-processamento

| Etapa | Conteúdo |
|------|----------|
| **0** | Imports e parâmetros |
| **1** | Leitura dos arquivos Intan (`.int`) + anotações |
| **2** | Remoção dos arquivos com comentário de falha |
| **3** | Detecção de ruído com SPARQ (MATLAB) |
| **4** | Filtro notch 55–65 Hz |
| **5** | Execução por rato |

## 0 — Imports e parâmetros

In [ ]:
import re
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import iirnotch, filtfilt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Aquisição
FS_ORIGINAL = 25_000   # Hz — frequência de aquisição do Intan
N_CANAIS    = 16

# Notch 55–65 Hz
NOTCH_LO     = 55.0
NOTCH_HI     = 65.0
NOTCH_CENTER = (NOTCH_LO + NOTCH_HI) / 2.0          # 60 Hz
NOTCH_Q      = NOTCH_CENTER / (NOTCH_HI - NOTCH_LO)  # Q = 6

# SPARQ
FS_SPARQ      = 1_000   # Hz — frequência do sinal exportado para o SPARQ
LIMIAR_RUIDO  = 0.50    # sessão descartada se a fração de amostras com ruído for maior que isso

# Caminhos
BASE_DIR         = Path.cwd()                                # raiz do repositório
DADOS_DIR        = BASE_DIR / 'dados_eletro'                 # contém as pastas R1, R2, ... (pode ser link simbólico)
XLSX_PATH        = BASE_DIR / 'data' / 'ALL_ANNOTATIONS.xlsx'
SPARQ_REPO       = BASE_DIR / 'SPARQ'
SPARQ_INPUT_DIR  = BASE_DIR / 'SPARQ_input'                  # .mat exportados para o SPARQ
SPARQ_RESULT_DIR = SPARQ_INPUT_DIR / 'SPARQ_results'         # onde o SPARQ salva os *_clean.mat
SPARQ_SCRIPT_DIR = BASE_DIR                                  # pasta com rodar_sparq.m e sparq_lfp.py
MATLAB_EXE       = 'matlab'                                  # ou caminho completo do matlab.exe

sys.path.insert(0, str(BASE_DIR))
from Downsample.read_intan_data import read_intan_data  # função padrão do lab
from sparq_lfp import (caminho_input_sparq, exportar_para_sparq,
                       localizar_resultado_sparq, mascara_sparq, abrir_sparq)

/mnt/dev/python/BioInfo-2026.2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 — Leitura dos arquivos Intan + anotações

Carrega as anotações de cada sessão, localiza os arquivos `.int` e faz o join pelo par rato / condição / trial.

In [ ]:
# ── 1.1 Anotações do Excel ───────────────────────────────────────────────────
dfs_anotacoes = []
xl = pd.ExcelFile(XLSX_PATH)
for sheet in xl.sheet_names:
    df_s = xl.parse(sheet)
    df_s.columns = [str(c).strip() for c in df_s.columns]
    if 'Rato' in df_s.columns:
        dfs_anotacoes.append(df_s[df_s['Rato'].notna()])

df_anot = pd.concat(dfs_anotacoes, ignore_index=True)
df_anot['Rato']           = df_anot['Rato'].astype(str).str.strip().str.upper()
df_anot['Condição']       = df_anot['Condição'].astype(str).str.strip()
df_anot['Sessão']         = df_anot['Sessão'].astype(str).str.strip().str.upper()
df_anot['acontencimento'] = df_anot.get('acontencimento', pd.Series(dtype=str)).fillna('')

print(f'Anotações carregadas: {len(df_anot)} linhas, {df_anot["Rato"].nunique()} ratos')

Anotações carregadas: 258 linhas, 10 ratos


In [ ]:
# ── 1.2 Arquivos .int ────────────────────────────────────────────────────────
def parse_nome_arquivo(stem):
    """Extrai rato, condição e trial do nome do arquivo.
    Aceita espaço, '_' ou nada como separador, com ou sem '%'.
    Exemplos: 'r1 25% A1_240416_124149' → R1, 0.25, A1
              'R1 25% T3_240416_132013' → R1, 0.25, T3
              'r1_25_T1_...'            → R1, 0.25, T1
              'R2D50A1_240429_122731'   → R2, 0.5, A1
              'R2D25%T4'                → R2, 0.25, T4
              'R3NORA2_...'             → R3, NOR, A2
    """
    stem_up = stem.strip().upper()

    m = re.match(r'(R\d+)[\s_]*D?(\d+)\s*%?[\s_]*([AT]\d+)', stem_up)
    if m:
        return m.group(1), str(int(m.group(2)) / 100), m.group(3)

    m = re.match(r'(R\d+)[\s_]*NOR[\s_]*([AT]\d+)', stem_up)
    if m:
        return m.group(1), 'NOR', m.group(2)

    return None, None, None


def rato_pela_pasta(path):
    """Usa o nome da pasta (ex.: dados_eletro/R1/...) quando o nome do arquivo não traz o rato."""
    for parte in reversed(path.parent.parts):
        m = re.fullmatch(r'R\d+', parte.strip().upper())
        if m:
            return m.group(0)
    return None


arquivos_int = sorted(p for p in DADOS_DIR.rglob('*') if p.suffix.lower() == '.int')

registros = []
for p in arquivos_int:
    rato, condicao, trial = parse_nome_arquivo(p.stem)
    registros.append({'path': p, 'stem': p.stem,
                      'rato': rato or rato_pela_pasta(p),
                      'condicao': condicao, 'trial': trial})

df_arquivos = pd.DataFrame(registros, columns=['path', 'stem', 'rato', 'condicao', 'trial'])
print(f'Pasta de dados: {DADOS_DIR}  (existe: {DADOS_DIR.exists()})')
print(f'Arquivos .int encontrados: {len(df_arquivos)}')
print(f'   Ratos: {sorted(df_arquivos["rato"].dropna().unique())}')

sem_padrao = df_arquivos[df_arquivos['trial'].isna()]
if not sem_padrao.empty:
    print(f'\n{len(sem_padrao)} arquivo(s) fora do padrão de sessão A/T ignorados (sono, modelagem, ...):')
    print(sem_padrao[['stem', 'rato']].to_string(index=False))
df_arquivos = df_arquivos[df_arquivos['trial'].notna()].reset_index(drop=True)
print(f'\nSessões A/T: {len(df_arquivos)}')

Pasta de dados: /mnt/dev/python/BioInfo-2026.2/dados_eletro  (existe: True)
Arquivos .int encontrados: 60
   Ratos: ['R2']

28 arquivo(s) fora do padrão de sessão A/T ignorados (sono, modelagem, ...):
                           stem rato
     R2SONOPOSD25_240501_131434   R2
     R2SONOPOSD25_240501_141434   R2
     R2SONOPOSD25_240501_141443   R2
     R2SONOPRED25_240501_100509   R2
     R2SONOPRED25_240501_110509   R2
     R2SONOPRED25_240501_110517   R2
       R2 POS D50_240429_132411   R2
       R2 POS D50_240429_142418   R2
  R2 PRE SONO 50%_240429_101839   R2
  R2 PRE SONO 50%_240429_111839   R2
       R2 POS 75%_240430_131452   R2
       R2 POS 75%_240430_141839   R2
R2 PRE 75% 2PARTE_240430_110933   R2
       R2 PRE 75%_240430_100847   R2
       R2 PRE 75%_240430_110847   R2
   POS NOR SONO 2_240428_134907   R2
   POS NOR SONO 2_240428_141056   R2
                 PRE NOR SONO 2   R2
          PRE NOR SONO_1 PART 2   R2
         PRE NOR SONO_1 part 1    R2
               R2_2404

In [ ]:
# ── 1.3 Join arquivos × anotações ────────────────────────────────────────────
df_anot['_key'] = (df_anot['Rato'] + '_' +
                   df_anot['Condição'].astype(str) + '_' +
                   df_anot['Sessão'])

df_arquivos['_key'] = (df_arquivos['rato'].fillna('') + '_' +
                       df_arquivos['condicao'].fillna('') + '_' +
                       df_arquivos['trial'].fillna(''))

df_arquivos = df_arquivos.merge(
    df_anot[['_key', 'acontencimento', 'Sono_pre', 'Sono_pos']].drop_duplicates('_key'),
    on='_key', how='left'
)
df_arquivos['acontencimento'] = df_arquivos['acontencimento'].fillna('')

print(df_arquivos[df_arquivos['acontencimento'] != '']
      [['stem', 'rato', 'condicao', 'trial', 'acontencimento']]
      .head(10).to_string(index=False))

                  stem rato condicao trial                      acontencimento
R2D25%A2_240501_122518   R2     0.25    A2                     Derrubou objeto
R2D25%A3_240501_124402   R2     0.25    A3 explorou mt pouco, demorou a entrar
R2D25%A4_240501_130101   R2     0.25    A4      falhou e tivemos que reiniciar
R2D25%T2_240501_123234   R2     0.25    T2       Derrubou objeto, não explorou
R2D25%T3_240501_125210   R2     0.25    T3               explorou bem no teste
 R2D50A4_240429_131240   R2      0.5    A4                     Derrubou objeto
 R2D75A1_240430_121435   R2     0.75    A1                      Roeu o objeto 
 R2D75A2_240430_123000   R2     0.75    A2                      Roeu o objeto 
 R2NORA2_240428_130711   R2      NOR    A2     cabo conectando e desconectando
 R2NORT1_240428_125837   R2      NOR    T1                     Derrubou objeto


In [ ]:
# ── 1.4 Leitura do sinal ─────────────────────────────────────────────────────
def carregar_intan(path):
    """Lê o .int e posiciona cada canal ativo na sua linha física (canal N → linha N-1).
    Canais inativos ficam zerados. Retorna (amps [N_CANAIS, N_amostras], canais_ativos)."""
    _, canais_ativos, data_neural, _ = read_intan_data(str(path))
    data_neural = np.asarray(data_neural, dtype=np.float32)

    amps = np.zeros((N_CANAIS, data_neural.shape[1]), dtype=np.float32)
    for pos, canal_fisico in enumerate(canais_ativos):
        if 1 <= canal_fisico <= N_CANAIS:
            amps[canal_fisico - 1] = data_neural[pos]
    return amps, canais_ativos

## 2 — Remoção dos arquivos com comentário de falha

In [ ]:
n_antes = len(df_arquivos)

mask_falha = df_arquivos['acontencimento'].str.lower().str.contains(
    'falha|cancelar|cancelada ', na=False
)
df_falhas = df_arquivos[mask_falha].copy()

print(f'Arquivos descartados por falha: {len(df_falhas)}')
if not df_falhas.empty:
    print(df_falhas[['stem', 'condicao', 'trial', 'acontencimento']].to_string(index=False))

df_arquivos = df_arquivos[~mask_falha].reset_index(drop=True)

print(f'\nArquivos restantes: {len(df_arquivos)} (removidos: {n_antes - len(df_arquivos)})')
print(f'   Ratos: {sorted(df_arquivos["rato"].dropna().unique())}')

Arquivos descartados por falha: 0

Arquivos restantes: 32 (removidos: 0)
   Ratos: ['R2']


## 3 — Detecção de ruído com SPARQ

O [SPARQ](https://github.com/lafise-ufrn/SPARQ) roda no MATLAB e exige a seleção manual de um trecho limpo de referência por sessão. O uso fica encapsulado em dois arquivos em `BASE_DIR`:

- `sparq_lfp.py` — exporta as sessões, abre o SPARQ e lê os resultados.
- `rodar_sparq.m` — abre a GUI do SPARQ já configurada para os `.mat` exportados (`LFP` canais × amostras, `fs`).

**Fluxo (duas passadas da célula de execução):**
1. Sessões sem resultado são exportadas para `SPARQ_input/<rato>/<stem>.mat` e o SPARQ é aberto no MATLAB.
2. Na GUI, marque o trecho limpo de cada sessão e salve todas. Os resultados vão para `SPARQ_input/SPARQ_results/.../<stem>_clean.mat` (localizados pelo padrão `<nome>*_clean.mat`).
3. Rodando a célula de novo, a máscara de ruído é aplicada.

**Observações:**
- **O SPARQ não identifica canais ruins.** A `noiseMask` é um vetor 1 × amostras, comum a todos os canais: marca *trechos de tempo* em que vários canais ultrapassam o limiar ao mesmo tempo. Por isso o critério aqui é por **sessão**: se a fração de amostras com ruído passar de `LIMIAR_RUIDO`, a sessão é descartada; caso contrário, as amostras marcadas viram `NaN` em todos os canais.
- **Exportação a `FS_SPARQ` = 1 kHz.** Os parâmetros do SPARQ são contados em amostras (ex.: intervalo de fusão de 100 amostras = 100 ms a 1 kHz, mas apenas 4 ms a 25 kHz). A máscara é depois expandida para `FS_ORIGINAL`. Se os parâmetros forem calibrados em outra frequência, altere `FS_SPARQ`.
- Os nomes dos `.mat` exportados têm espaços e `%` trocados por `_` (o SPARQ faz o mesmo ao salvar), ex.: `r1 25% A1_...` → `r1_25__A1_...`.
- **Resultados em `.mat` v7.3**, que o `scipy.io.loadmat` não lê; a leitura usa `h5py` (`pip install h5py`).
- **Notch antes do `NaN`**: o `filtfilt` propaga `NaN` pelo canal inteiro, então a ordem é máscara → notch → amostras de ruído viram `NaN`.
- `MATLAB_EXE` precisa estar no PATH (ou ser o caminho completo do `matlab.exe`).

## 4 — Filtro notch 55–65 Hz

Remove a interferência da rede elétrica (60 Hz) com `iirnotch` + `filtfilt` (fase zero), aplicado na `FS_ORIGINAL`.

In [ ]:
def aplicar_notch(data_neural, fs):
    """Aplica filtro notch centrado em 60 Hz (55–65 Hz) em todos os canais."""
    b, a = iirnotch(w0=NOTCH_CENTER, Q=NOTCH_Q, fs=fs)
    data_filtrado = np.zeros_like(data_neural, dtype=float)
    for ch in range(data_neural.shape[0]):
        data_filtrado[ch] = filtfilt(b, a, data_neural[ch].astype(float))
    return data_filtrado


print(f'Notch: {NOTCH_LO}–{NOTCH_HI} Hz | centro={NOTCH_CENTER} Hz | Q={NOTCH_Q:.1f}')

Notch: 55.0–65.0 Hz | centro=60.0 Hz | Q=6.0


## 5 — Execução

Defina `RATO_PIPELINE` (`'R1'`, `'R3'`, ... ou `'ALL'`) e rode a célula.

- Sessões **sem resultado SPARQ** são exportadas e ficam como `PENDENTE_SPARQ`; com `ABRIR_SPARQ = True` o SPARQ é aberto no MATLAB. Salve os resultados e execute a célula de novo.
- Sessões **com resultado SPARQ** passam por máscara de ruído + notch.

In [ ]:
# ══════════════════════════════════════════════════════════════
# SELETOR DE RATO — 'R1', 'R3', ... ou 'ALL'
# ══════════════════════════════════════════════════════════════
RATO_PIPELINE = 'R2'   # ← altere aqui
ABRIR_SPARQ   = True    # abre o SPARQ no MATLAB se houver sessões pendentes

_rato = RATO_PIPELINE.strip().upper()
if _rato == 'ALL':
    df_run = df_arquivos.copy()
else:
    df_run = df_arquivos[df_arquivos['rato'].str.upper() == _rato].copy()
    if df_run.empty:
        raise ValueError(f"Rato '{RATO_PIPELINE}' não encontrado. "
                         f"Disponíveis: {sorted(df_arquivos['rato'].dropna().unique())}")
print(f'[{_rato}] {len(df_run)} arquivos de {df_run["rato"].nunique()} rato(s)')

relatorio = []

for _, row in tqdm(df_run.iterrows(), total=len(df_run), desc=f'Processando [{_rato}]'):
    stem, rato = row['stem'], row['rato']
    base = {'stem': stem, 'rato': rato, 'condicao': row['condicao'], 'trial': row['trial']}
    try:
        # 3. SPARQ — exporta se ainda não houver resultado
        if localizar_resultado_sparq(SPARQ_RESULT_DIR, stem) is None:
            if not caminho_input_sparq(SPARQ_INPUT_DIR, rato, stem).exists():
                amps, _ = carregar_intan(row['path'])
                exportar_para_sparq(amps, FS_ORIGINAL, FS_SPARQ, SPARQ_INPUT_DIR, rato, stem)
                del amps
                gc.collect()
            relatorio.append({**base, 'status': 'PENDENTE_SPARQ'})
            continue

        # 1. Leitura
        amps, canais_ativos = carregar_intan(row['path'])

        # 3. Máscara de ruído do SPARQ
        mask = mascara_sparq(SPARQ_RESULT_DIR, stem, amps.shape[1], FS_SPARQ, FS_ORIGINAL)
        frac_ruido = float(mask.mean())
        if frac_ruido > LIMIAR_RUIDO:
            relatorio.append({**base, 'status': 'DESCARTADO', 'frac_ruido': frac_ruido})
            del amps
            gc.collect()
            continue

        # 4. Notch (antes do NaN, pois filtfilt propaga NaN pelo canal inteiro)
        amps = aplicar_notch(amps, FS_ORIGINAL)
        amps[:, mask] = np.nan

        # TODO: próximas etapas

        relatorio.append({**base, 'status': 'OK', 'frac_ruido': frac_ruido})
        del amps, mask
        gc.collect()

    except Exception as e:
        relatorio.append({**base, 'status': 'ERRO', 'motivo': str(e)})
        print(f'ERRO em {stem}: {e}')

df_relatorio = pd.DataFrame(relatorio)
print(df_relatorio.groupby('status').size().to_string())

if (df_relatorio['status'] == 'PENDENTE_SPARQ').any():
    print(f'\nSessões aguardando SPARQ em: {SPARQ_INPUT_DIR}')
    if ABRIR_SPARQ:
        abrir_sparq(SPARQ_REPO, SPARQ_INPUT_DIR, SPARQ_SCRIPT_DIR, MATLAB_EXE)
        print('SPARQ aberto no MATLAB. Salve os resultados e execute esta célula novamente.')

[R2] 32 arquivos de 1 rato(s)



Processando [R2]:   0%|          | 0/32 [00:00<?, ?it/s]


Data file contains 195.93 seconds of data from 16 amplifier channels.
Channels: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16



Processando [R2]:   3%|▎         | 1/32 [00:05<02:43,  5.29s/it]


Data file contains 190.59 seconds of data from 16 amplifier channels.
Channels: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16


...


Processando [R2]:  97%|█████████▋| 31/32 [02:35<00:04,  4.31s/it]


Data file contains 191.40 seconds of data from 16 amplifier channels.
Channels: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16



Processando [R2]: 100%|██████████| 32/32 [02:39<00:00,  4.24s/it]


Processando [R2]: 100%|██████████| 32/32 [02:39<00:00,  4.98s/it]

status
DESCARTADO     4
OK            28
